In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from results_utils import DATA_DIR, RESULTS_DIR, callbacks_paths, save_results

tf.keras.utils.set_random_seed(42)

MODEL_NAME = "cnn_model_jean"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 60
LEARNING_RATE = 1e-3
PATIENCE = 8
FILTERS = (32, 64, 128, 256)
DROPOUT = 0.5
DENSE_UNITS = 128
AUGMENT = True
CLASS_WEIGHT = True

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "valid",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

class_names = train_ds.class_names

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=42).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
counts = np.array([len(list((DATA_DIR / "train" / c).glob("*.jpg"))) for c in class_names])
class_weight = {i: counts.sum() / (len(counts) * n) for i, n in enumerate(counts)} if CLASS_WEIGHT else None

pd.DataFrame({"class": class_names, "images": counts, "weight": [round(counts.sum() / (len(counts) * n), 3) for n in counts]})

In [ ]:
def build_cnn_model(input_shape=(224, 224, 3), num_classes=10, filters=(32, 64, 128, 256),
                    dropout=0.5, dense_units=128, augment=True):
    model = tf.keras.Sequential(name=MODEL_NAME)
    model.add(tf.keras.layers.Input(shape=input_shape))

    if augment:
        model.add(tf.keras.layers.RandomFlip("horizontal"))
        model.add(tf.keras.layers.RandomRotation(0.1))
        model.add(tf.keras.layers.RandomZoom(0.1))

    model.add(tf.keras.layers.Rescaling(1./255))

    for f in filters:
        model.add(tf.keras.layers.Conv2D(filters=f, kernel_size=(3,3), padding='same', kernel_initializer='he_uniform', use_bias=False))
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Activation('relu'))
        model.add(tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=2))

    model.add(tf.keras.layers.GlobalAveragePooling2D())
    model.add(tf.keras.layers.Dropout(dropout))
    model.add(tf.keras.layers.Dense(dense_units, activation=tf.nn.relu, kernel_initializer='he_uniform'))
    model.add(tf.keras.layers.Dropout(dropout / 2))
    model.add(tf.keras.layers.Dense(num_classes, activation=tf.nn.softmax))

    return model

cnn_model = build_cnn_model(input_shape=(*IMG_SIZE, 3), num_classes=len(class_names), filters=FILTERS,
                            dropout=DROPOUT, dense_units=DENSE_UNITS, augment=AUGMENT)
cnn_model.summary()

In [ ]:
cnn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

best_model_path, history_path = callbacks_paths(MODEL_NAME)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint(best_model_path, monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.CSVLogger(history_path)
]

history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks
)

In [ ]:
params = {"epochs": EPOCHS, "lr": LEARNING_RATE, "patience": PATIENCE, "batch_size": BATCH_SIZE,
          "filters": str(FILTERS), "dropout": DROPOUT, "dense_units": DENSE_UNITS,
          "augment": AUGMENT, "batchnorm": True, "class_weight": CLASS_WEIGHT}

true_indices, predicted_indices, predictions, test_acc = save_results(MODEL_NAME, cnn_model, test_ds, class_names, params)
print(f"Test accuracy: {test_acc:.2%}")

In [ ]:
hist = pd.read_csv(history_path)
best_epoch = hist["val_loss"].idxmin()

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(hist["accuracy"], label='Train Accuracy')
plt.plot(hist["val_accuracy"], label='Val Accuracy')
plt.axvline(best_epoch, color='gray', linestyle='--')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(hist["loss"], label='Train Loss')
plt.plot(hist["val_loss"], label='Val Loss')
plt.axvline(best_epoch, color='gray', linestyle='--')
plt.legend()
plt.title('Loss')
plt.show()

In [ ]:
cm = confusion_matrix(true_indices, predicted_indices)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.title(f"{MODEL_NAME} - test accuracy {test_acc:.0%}")
plt.tight_layout()
plt.show()

In [ ]:
test_images = np.concatenate([images.numpy() for images, _ in test_ds], axis=0)
confidences = predictions.max(axis=1)
wrong = np.where(predicted_indices != true_indices)[0]
rows = max(1, int(np.ceil(len(wrong) / 5)))

fig, axes = plt.subplots(rows, 5, figsize=(16, 3.5 * rows), squeeze=False)
axes = axes.flatten()

for ax in axes:
    ax.axis("off")

for ax, i in zip(axes, wrong):
    ax.imshow(test_images[i].astype("uint8"))
    ax.set_title(f"Pred: {class_names[predicted_indices[i]]}\n(True: {class_names[true_indices[i]]})\n{confidences[i] * 100:.0f}%", color="red", fontsize=9)

plt.suptitle(f"Misclassified test images: {len(wrong)} / {len(true_indices)}")
plt.tight_layout()
plt.show()